# DiDAQt FABRIC Experiment

This notebook creates a FABRIC testbed artifact that demonstrates the DiDAQt
fault-detection framework for DAQ networks.

**Topology:**
- 1 Sender node (ConnectX-6, 2 ports) — runs 10 sender instances
- 1 Receiver node (ConnectX-6, 2 ports + 2 NIC_Basic) — runs 2 receiver instances
- 1 Controller node (NIC_Basic) — runs the heartbeat monitor
- 1 Tofino P4 switch — L2 MAC forwarding with VLAN rewriting

**Data path (in-band):** Sender ports ↔ Tofino ↔ Receiver ports (VLAN-tagged)
- Cross-site links use **L2PTP**; same-site links use **L2Bridge**.

**Control path (out-of-band):** Receiver NICs → Controller (FABNetv4 / L3)

## 1. Setup

In [1]:
from fabrictestbed_extensions.fablib.fablib import FablibManager

fablib = FablibManager()

User: awolosewicz@hawk.iit.edu bastion key is valid!
Configuration is valid


Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
Token File,/home/fabric/.tokens.json
Project ID,abe55161-26d9-434a-826a-4f9a655d0dde
Bastion Host,bastion.fabric-testbed.net
Bastion Username,awolosewicz_0000144375
Bastion Private Key File,/home/fabric/work/fabric_config/bastion3
Slice Public Key File,/home/fabric/work/fabric_config/fpga-silver.pub
Slice Private Key File,/home/fabric/work/fabric_config/fpga-silver


Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
Token File,/home/fabric/.tokens.json
Project ID,abe55161-26d9-434a-826a-4f9a655d0dde
Bastion Host,bastion.fabric-testbed.net
Bastion Username,awolosewicz_0000144375
Bastion Private Key File,/home/fabric/work/fabric_config/bastion3
Slice Public Key File,/home/fabric/work/fabric_config/fpga-silver.pub
Slice Private Key File,/home/fabric/work/fabric_config/fpga-silver


In [ ]:
fablib.get_random_site(filter_function=lambda x: x["p4-switch_available"] > 0 and x["nic_connectx_6_available"] > 1 and 

In [9]:
fablib.list_sites(filter_function=lambda x: x["p4-switch_available"] > 0 and x["nic_connectx_6_available"] > 1, pretty_names=False)

name,state,address,location,ptp_capable,hosts,cpus,cores_available,cores_capacity,cores_allocated,ram_available,ram_capacity,ram_allocated,disk_available,disk_capacity,disk_allocated,nic_basic_available,nic_basic_capacity,nic_basic_allocated,p4-switch_available,p4-switch_capacity,p4-switch_allocated,nic_connectx_6_available,nic_connectx_6_capacity,nic_connectx_6_allocated,nic_connectx_5_available,nic_connectx_5_capacity,nic_connectx_5_allocated,nic_connectx_7_100_available,nic_connectx_7_100_capacity,nic_connectx_7_100_allocated,nic_connectx_7_400_available,nic_connectx_7_400_capacity,nic_connectx_7_400_allocated,nic_bluefield2_connectx_5_available,nic_bluefield2_connectx_5_capacity,nic_bluefield2_connectx_5_allocated,nvme_available,nvme_capacity,nvme_allocated,tesla_t4_available,tesla_t4_capacity,tesla_t4_allocated,rtx6000_available,rtx6000_capacity,rtx6000_allocated,a30_available,a30_capacity,a30_allocated,a40_available,a40_capacity,a40_allocated,fpga_u280_available,fpga_u280_capacity,fpga_u280_allocated,fpga_sn1022_available,fpga_sn1022_capacity,fpga_sn1022_allocated
UTAH,Active,"875 South West Temple,Salt Lake City, UT 84101","(40.7503666, -111.893838)",True,5,10,152,640,488,1190,2390,1200,98051,107461,9410,395,635,240,1,1,0,2,2,0,4,4,0,0,0,0,0,0,0,0,0,0,16,16,0,1,4,3,0,6,6,0,0,0,0,0,0,1,1,0,0,0,0


name,state,address,location,ptp_capable,hosts,cpus,cores_available,cores_capacity,cores_allocated,ram_available,ram_capacity,ram_allocated,disk_available,disk_capacity,disk_allocated,nic_basic_available,nic_basic_capacity,nic_basic_allocated,p4-switch_available,p4-switch_capacity,p4-switch_allocated,nic_connectx_6_available,nic_connectx_6_capacity,nic_connectx_6_allocated,nic_connectx_5_available,nic_connectx_5_capacity,nic_connectx_5_allocated,nic_connectx_7_100_available,nic_connectx_7_100_capacity,nic_connectx_7_100_allocated,nic_connectx_7_400_available,nic_connectx_7_400_capacity,nic_connectx_7_400_allocated,nic_bluefield2_connectx_5_available,nic_bluefield2_connectx_5_capacity,nic_bluefield2_connectx_5_allocated,nvme_available,nvme_capacity,nvme_allocated,tesla_t4_available,tesla_t4_capacity,tesla_t4_allocated,rtx6000_available,rtx6000_capacity,rtx6000_allocated,a30_available,a30_capacity,a30_allocated,a40_available,a40_capacity,a40_allocated,fpga_u280_available,fpga_u280_capacity,fpga_u280_allocated,fpga_sn1022_available,fpga_sn1022_capacity,fpga_sn1022_allocated
UTAH,Active,"875 South West Temple,Salt Lake City, UT 84101","(40.7503666, -111.893838)",True,5,10,152,640,488,1190,2390,1200,98051,107461,9410,395,635,240,1,1,0,2,2,0,4,4,0,0,0,0,0,0,0,0,0,0,16,16,0,1,4,3,0,6,6,0,0,0,0,0,0,1,1,0,0,0,0


In [10]:
# ---------- Configuration ----------
# Each component can be placed on a separate FABRIC site.
# Cross-site in-band links use L2PTP; same-site links use L2Bridge.
# Run fablib.list_sites() to find sites with Tofino switches and ConnectX-6 NICs.

SENDER_SITE     = 'UTAH'
RECEIVER_SITE   = 'UTAH'
SWITCH_SITE     = 'UTAH'
CONTROLLER_SITE = 'UTAH'

SLICE_NAME = 'didaqt-experiment'
IMAGE      = 'default_ubuntu_22'

## 2. Create Topology

In [11]:
slice = fablib.new_slice(name=SLICE_NAME)

# ---- Nodes ----
sender_node = slice.add_node(name='sender', site=SENDER_SITE,
                             cores=8, ram=32, disk=20, image=IMAGE)
receiver_node = slice.add_node(name='receiver', site=RECEIVER_SITE,
                               cores=8, ram=32, disk=20, image=IMAGE)
controller_node = slice.add_node(name='controller', site=CONTROLLER_SITE,
                                 cores=4, ram=8, disk=20, image=IMAGE)

# ---- P4 Switch ----
p4_switch = slice.add_switch(name='p4_switch', site=SWITCH_SITE)

# ---- In-band NICs (ConnectX-6, dual-port 100G) ----
sender_nic = sender_node.add_component(model='NIC_ConnectX_6', name='sender_nic')
rx_nic     = receiver_node.add_component(model='NIC_ConnectX_6', name='rx_nic')

sender_ifaces = sender_nic.get_interfaces()
rx_ifaces     = rx_nic.get_interfaces()
sw_ifaces     = p4_switch.get_interfaces()

print(f'Sender NIC interfaces:   {[i.get_name() for i in sender_ifaces]}')
print(f'Receiver NIC interfaces: {[i.get_name() for i in rx_ifaces]}')
print(f'Switch interfaces:       {[i.get_name() for i in sw_ifaces]}')

# ---- In-band L2 networks (sender/receiver <-> switch) ----
# Use L2PTP when the two endpoints are on different sites, L2Bridge when same-site.

def l2_type(site_a, site_b):
    return 'L2PTP' if site_a != site_b else 'L2Bridge'

l2_type_sender_sw = l2_type(SENDER_SITE, SWITCH_SITE)
l2_type_rx_sw     = l2_type(RECEIVER_SITE, SWITCH_SITE)

print(f'\nSender  <-> Switch link type: {l2_type_sender_sw}')
print(f'Receiver <-> Switch link type: {l2_type_rx_sw}')

# Sender port 0 <-> Switch port 0
net_s0 = slice.add_l2network(name='net-s0-sw',
                             interfaces=[sender_ifaces[0], sw_ifaces[0]],
                             type=l2_type_sender_sw)
# Sender port 1 <-> Switch port 1
net_s1 = slice.add_l2network(name='net-s1-sw',
                             interfaces=[sender_ifaces[1], sw_ifaces[1]],
                             type=l2_type_sender_sw)
# Receiver port 0 <-> Switch port 2
net_r0 = slice.add_l2network(name='net-r0-sw',
                             interfaces=[rx_ifaces[0], sw_ifaces[2]],
                             type=l2_type_rx_sw)
# Receiver port 1 <-> Switch port 3
net_r1 = slice.add_l2network(name='net-r1-sw',
                             interfaces=[rx_ifaces[1], sw_ifaces[3]],
                             type=l2_type_rx_sw)

# ---- Out-of-band NICs (NIC_Basic for heartbeat L3 network) ----
rx_ctrl_nic0 = receiver_node.add_component(model='NIC_Basic', name='rx_ctrl0')
rx_ctrl_nic1 = receiver_node.add_component(model='NIC_Basic', name='rx_ctrl1')
ctrl_nic     = controller_node.add_component(model='NIC_Basic', name='ctrl_nic')

ctrl_net = slice.add_l3network(name='ctrl-net', interfaces=[
    rx_ctrl_nic0.get_interfaces()[0],
    rx_ctrl_nic1.get_interfaces()[0],
    ctrl_nic.get_interfaces()[0]
], type='IPv4')

print('\nTopology defined.')
slice.show()

Sender NIC interfaces:   ['sender-sender_nic-p1', 'sender-sender_nic-p2']
Receiver NIC interfaces: ['receiver-rx_nic-p1', 'receiver-rx_nic-p2']
Switch interfaces:       ['p1', 'p2', 'p3', 'p4', 'p5', 'p6', 'p7', 'p8']

Sender  <-> Switch link type: L2Bridge
Receiver <-> Switch link type: L2Bridge

Topology defined.


ID,None
Name,didaqt-experiment
Lease Expiration (UTC),None
Lease Start (UTC),None
Project ID,None
State,None
Email,None
UserId,None


ID,None
Name,didaqt-experiment
Lease Expiration (UTC),None
Lease Start (UTC),None
Project ID,None
State,None
Email,None
UserId,None


## 3. Submit Slice

In [12]:
slice.submit()
print('Slice submitted and ready.')


Retry: 32, Time: 739 sec


ID,1eb1133e-9ab9-42a8-bb78-ce123dc93a0d
Name,didaqt-experiment
Lease Expiration (UTC),2026-03-22 22:33:31 +0000
Lease Start (UTC),2026-03-21 22:33:31 +0000
Project ID,abe55161-26d9-434a-826a-4f9a655d0dde
State,StableOK
Email,awolosewicz@hawk.illinoistech.edu
UserId,30a258a1-6bed-43e7-a6bc-29d00bcaeb55


ID,Name,Cores,RAM,Disk,Image,Image Type,Host,Site,Username,Management IP,State,Error,SSH Command,Public SSH Key File,Private SSH Key File
46d8cbca-bab0-4486-8d74-5bfcee31ccb3,controller,4,8,100,default_ubuntu_22,qcow2,utah-w5.fabric-testbed.net,UTAH,ubuntu,2001:1948:417:7:f816:3eff:fe38:1a14,Active,,ssh -i /home/fabric/work/fabric_config/fpga-silver -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:1948:417:7:f816:3eff:fe38:1a14,/home/fabric/work/fabric_config/fpga-silver.pub,/home/fabric/work/fabric_config/fpga-silver
8375b0d8-56e4-4536-a6ee-93dc95da6ab0,p4_switch,,,,,,,UTAH,fabric,2001:1948:417:7:290:fbff:fe76:d00e,Active,,ssh -i /home/fabric/work/fabric_config/fpga-silver -F /home/fabric/work/fabric_config/ssh_config fabric@2001:1948:417:7:290:fbff:fe76:d00e,/home/fabric/work/fabric_config/fpga-silver.pub,/home/fabric/work/fabric_config/fpga-silver
7a622411-c0ca-4e58-8bdd-4bf810c0c41b,receiver,8,32,100,default_ubuntu_22,qcow2,utah-w3.fabric-testbed.net,UTAH,ubuntu,2001:1948:417:7:f816:3eff:fe30:84ac,Active,,ssh -i /home/fabric/work/fabric_config/fpga-silver -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:1948:417:7:f816:3eff:fe30:84ac,/home/fabric/work/fabric_config/fpga-silver.pub,/home/fabric/work/fabric_config/fpga-silver
a3164a0c-6116-4a17-aaf1-8db52d27f05f,sender,8,32,100,default_ubuntu_22,qcow2,utah-w3.fabric-testbed.net,UTAH,ubuntu,2001:1948:417:7:f816:3eff:fe3a:8ef4,Active,,ssh -i /home/fabric/work/fabric_config/fpga-silver -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:1948:417:7:f816:3eff:fe3a:8ef4,/home/fabric/work/fabric_config/fpga-silver.pub,/home/fabric/work/fabric_config/fpga-silver


ID,Name,Layer,Type,Site,Subnet,Gateway,State,Error
0431bfc0-8523-4fed-ac94-af90f6c60515,ctrl-net,L3,FABNetv4,UTAH,10.132.10.0/24,10.132.10.1,Active,
ec322bb8-7f3f-42c6-9aba-5098b363045f,net-r0-sw,L2,L2Bridge,UTAH,None,None,Active,
47309994-25e1-4449-9c5c-98e37574ce3c,net-r1-sw,L2,L2Bridge,UTAH,None,None,Active,
e2ad0349-491e-4958-92df-1583511061a9,net-s0-sw,L2,L2Bridge,UTAH,None,None,Active,
50c53ea7-a197-43a6-89e3-0d96e6f9547e,net-s1-sw,L2,L2Bridge,UTAH,None,None,Active,


Name,Short Name,Node,Network,Bandwidth,Mode,VLAN,MAC,Physical Device,Device,IP Address,Numa Node,Switch Port
sender-sender_nic-p1,p1,sender,net-s0-sw,100,config,,04:3F:72:FE:A6:30,enp7s0np0,enp7s0np0,fe80::63f:72ff:fefe:a630,1,HundredGigE0/0/0/0
sender-sender_nic-p2,p2,sender,net-s1-sw,100,config,,04:3F:72:FE:A6:31,enp8s0np0,enp8s0np0,fe80::63f:72ff:fefe:a631,1,HundredGigE0/0/0/2
receiver-rx_nic-p1,p1,receiver,net-r0-sw,100,config,,04:3F:72:FE:A6:20,enp8s0np0,enp8s0np0,fe80::63f:72ff:fefe:a620,6,HundredGigE0/0/0/4
receiver-rx_nic-p2,p2,receiver,net-r1-sw,100,config,,04:3F:72:FE:A6:21,enp9s0np0,enp9s0np0,fe80::63f:72ff:fefe:a621,6,HundredGigE0/0/0/6
receiver-rx_ctrl0-p1,p1,receiver,ctrl-net,100,config,,FA:67:EC:54:AB:E2,enp10s0,enp10s0,fe80::f867:ecff:fe54:abe2,4,HundredGigE0/0/0/9
receiver-rx_ctrl1-p1,p1,receiver,ctrl-net,100,config,,FA:3E:33:11:57:11,enp7s0,enp7s0,fe80::f83e:33ff:fe11:5711,4,HundredGigE0/0/0/9
controller-ctrl_nic-p1,p1,controller,ctrl-net,100,config,,BE:25:78:07:31:80,enp7s0,enp7s0,fe80::bc25:78ff:fe07:3180,4,HundredGigE0/0/0/13
p1,p1,p4_switch,net-s0-sw,100,config,,None,None,1,None,None,HundredGigE0/0/0/0
p2,p2,p4_switch,net-s1-sw,100,config,,None,None,2,None,None,HundredGigE0/0/0/2
p3,p3,p4_switch,net-r0-sw,100,config,,None,None,3,None,None,HundredGigE0/0/0/24/2



Time to print interfaces 764 seconds
Slice submitted and ready.


## 4. Gather Topology Information

After the slice is provisioned, query the actual OS interface names,
MAC addresses, VLAN IDs, and L3 IP addresses.

In [13]:
slice = fablib.get_slice(name=SLICE_NAME)

sender_node     = slice.get_node('sender')
receiver_node   = slice.get_node('receiver')
controller_node = slice.get_node('controller')
p4_switch       = slice.get_node('p4_switch')

def vlan_or_zero(iface):
    """Return the VLAN ID as an int, or 0 if the link has no VLAN (L2Bridge)."""
    v = iface.get_vlan()
    return int(v) if v else 0

# ---- Sender in-band interfaces ----
s_iface0 = sender_node.get_interface(network_name='net-s0-sw')
s_iface1 = sender_node.get_interface(network_name='net-s1-sw')

s0_os   = s_iface0.get_physical_os_interface_name()
s0_mac  = s_iface0.get_mac()
s0_vlan = vlan_or_zero(s_iface0)
s1_os   = s_iface1.get_physical_os_interface_name()
s1_mac  = s_iface1.get_mac()
s1_vlan = vlan_or_zero(s_iface1)

print(f'Sender port 0: iface={s0_os}  mac={s0_mac}  vlan={s0_vlan}')
print(f'Sender port 1: iface={s1_os}  mac={s1_mac}  vlan={s1_vlan}')

# ---- Receiver in-band interfaces ----
r_iface0 = receiver_node.get_interface(network_name='net-r0-sw')
r_iface1 = receiver_node.get_interface(network_name='net-r1-sw')

r0_os   = r_iface0.get_physical_os_interface_name()
r0_mac  = r_iface0.get_mac()
r0_vlan = vlan_or_zero(r_iface0)
r1_os   = r_iface1.get_physical_os_interface_name()
r1_mac  = r_iface1.get_mac()
r1_vlan = vlan_or_zero(r_iface1)

print(f'\nReceiver port 0: iface={r0_os}  mac={r0_mac}  vlan={r0_vlan}')
print(f'Receiver port 1: iface={r1_os}  mac={r1_mac}  vlan={r1_vlan}')

# ---- Switch ports for each L2 network ----
sw_iface_s0 = p4_switch.get_interface(network_name='net-s0-sw')
sw_iface_s1 = p4_switch.get_interface(network_name='net-s1-sw')
sw_iface_r0 = p4_switch.get_interface(network_name='net-r0-sw')
sw_iface_r1 = p4_switch.get_interface(network_name='net-r1-sw')

sw_port_s0 = sw_iface_s0.get_device_name()
sw_port_s1 = sw_iface_s1.get_device_name()
sw_port_r0 = sw_iface_r0.get_device_name()
sw_port_r1 = sw_iface_r1.get_device_name()

sw_vlan_s0 = vlan_or_zero(sw_iface_s0)
sw_vlan_s1 = vlan_or_zero(sw_iface_s1)
sw_vlan_r0 = vlan_or_zero(sw_iface_r0)
sw_vlan_r1 = vlan_or_zero(sw_iface_r1)

print(f'\nSwitch port for sender0:   {sw_port_s0}  vlan={sw_vlan_s0}')
print(f'Switch port for sender1:   {sw_port_s1}  vlan={sw_vlan_s1}')
print(f'Switch port for receiver0: {sw_port_r0}  vlan={sw_vlan_r0}')
print(f'Switch port for receiver1: {sw_port_r1}  vlan={sw_vlan_r1}')

if s0_vlan == 0:
    print('\nNote: VLAN IDs are 0 (L2Bridge / same-site). Frames will be untagged.')
else:
    print(f'\nVLAN tagging is active (L2PTP / cross-site).')

# ---- Controller L3 interface ----
ctrl_iface = controller_node.get_interface(network_name='ctrl-net')
ctrl_ip = ctrl_iface.get_ip_addr()

rx_ctrl_iface0 = receiver_node.get_interface(network_name='ctrl-net')
# The receiver has two NIC_Basic on ctrl-net; get both IPs.
rx_ctrl_ifaces = [i for i in receiver_node.get_interfaces()
                  if i.get_network() and i.get_network().get_name() == 'ctrl-net']
rx_ctrl_ips = [i.get_ip_addr() for i in rx_ctrl_ifaces]

print(f'\nController IP: {ctrl_ip}')
print(f'Receiver ctrl IPs: {rx_ctrl_ips}')

Sender port 0: iface=enp7s0np0  mac=04:3F:72:FE:A6:30  vlan=0
Sender port 1: iface=enp8s0np0  mac=04:3F:72:FE:A6:31  vlan=0

Receiver port 0: iface=enp8s0np0  mac=04:3F:72:FE:A6:20  vlan=0
Receiver port 1: iface=enp9s0np0  mac=04:3F:72:FE:A6:21  vlan=0

Switch port for sender0:   1  vlan=0
Switch port for sender1:   2  vlan=0
Switch port for receiver0: 3  vlan=0
Switch port for receiver1: 4  vlan=0

Note: VLAN IDs are 0 (L2Bridge / same-site). Frames will be untagged.

Controller IP: fe80::bc25:78ff:fe07:3180
Receiver ctrl IPs: ['fe80::f867:ecff:fe54:abe2', 'fe80::f83e:33ff:fe11:5711']


## 5. Install Dependencies

In [14]:
install_cmd = 'sudo apt-get update -qq && sudo apt-get install -y -qq build-essential ethtool'

from concurrent.futures import ThreadPoolExecutor

def install_on(node):
    name = node.get_name()
    print(f'Installing on {name}...')
    stdout, stderr = node.execute(install_cmd, quiet=True)
    print(f'  {name}: done')
    return stdout, stderr

with ThreadPoolExecutor(max_workers=3) as pool:
    futures = [pool.submit(install_on, n)
               for n in [sender_node, receiver_node, controller_node]]
    for f in futures:
        f.result()

print('All dependencies installed.')

Installing on sender...
Installing on receiver...
Installing on controller...
  controller: done
  receiver: done
  sender: done
All dependencies installed.


## 6. Upload Source Code

In [ ]:
import os

# Paths relative to this notebook (artifact/)
REPO = os.path.abspath('..')

for node in [controller_node, sender_node, receiver_node]:
    node.upload_directory(REPO, "/home/ubuntu/")

# Switch — only the P4 program
p4_switch.execute(f'mkdir -p examples/p4', quiet=True)
p4_switch.upload_file(
    os.path.join(REPO, 'examples/p4/l2_forward.p4'),
    f'examples/p4/l2_forward.p4'
)
print(f'  p4_switch: P4 program uploaded')

print('Upload complete.')

## 7. Compile

In [28]:
# Sender: just needs gcc
print('Compiling sender...')
sender_node.execute(
    f'cd {REMOTE_DIR} && make examples',
    quiet=True
)
print('  sender: OK')

# Receiver: needs libdidaqt + receiver
print('Compiling receiver...')
receiver_node.execute(
    f'cd {REMOTE_DIR} && make examples',
    quiet=True
)
print('  receiver: OK')

# Controller: heartbeat monitor
print('Compiling heartbeat_monitor...')
controller_node.execute(
    f'cd {REMOTE_DIR} && make examples',
    quiet=True
)
print('  heartbeat_monitor: OK')

print('All binaries compiled.')

Compiling sender...
  sender: OK
Compiling receiver...
  receiver: OK
Compiling heartbeat_monitor...
  heartbeat_monitor: OK
All binaries compiled.


## 8. Configure P4 Switch

Compile the P4 program on the Tofino, start `bf_switchd`, and
install the initial L2 forwarding rules.

**Note:** The Barefoot SDE path (`$SDE`) varies by FABRIC site.
Adjust the variable below if needed.

In [19]:
SDE = '/opt/bf-sde'  # Adjust if your FABRIC site uses a different path
P4_SRC = f'examples/p4/l2_forward.p4'
P4_PROG = 'l2_forward'

# Compile the P4 program
print('Compiling P4 program on switch...')
stdout, stderr = p4_switch.execute(
    f'cd {SDE} && ./p4_build.sh {P4_SRC}',
    quiet=True, timeout=300
)
print('  P4 compilation done.')

# Start bf_switchd in the background
print('Starting bf_switchd...')
p4_switch.execute(
    f'nohup {SDE}/run_switchd.sh -p {P4_PROG} '
    f'> /tmp/switchd.log 2>&1 &',
    quiet=True
)

import time
time.sleep(15)  # Wait for bf_switchd to initialize
print('  bf_switchd started (log at /tmp/switchd.log on switch).')

Compiling P4 program on switch...
  P4 compilation done.
Starting bf_switchd...
  bf_switchd started (log at /tmp/switchd.log on switch).


In [20]:
# Build a bfrt_python script that installs initial forwarding rules.
#
# The rule maps the receiver's destination MAC to the correct switch
# egress port and rewrites the VLAN ID for the receiver-side link.
# When using L2Bridge (same-site, no VLAN), vlan_id is 0 — the P4
# forward action writes hdr.vlan.vid but the header is invalid so
# the write is a no-op and no tag is emitted.
#
# NOTE: sw_port values are FABRIC device names (e.g., '1/0').
# The Tofino's internal dev_port numbering may differ. Query with:
#   bfrt> bfrt.port.port_hdl_info.get(CONN_ID=x, CHNL_ID=y)
# Adjust the port numbers below after checking the switch.

def mac_to_int(mac_str):
    """Convert 'AA:BB:CC:DD:EE:FF' to integer."""
    return int(mac_str.replace(':', ''), 16)

# We need the dev_port (integer) for each switch port.
# These may need adjustment — query the switch if the defaults are wrong.
# For now, we use the FABRIC device names and try to extract port numbers.
print('Switch port mapping (verify these on the switch):')
print(f'  sender0   -> switch port: {sw_port_s0}')
print(f'  sender1   -> switch port: {sw_port_s1}')
print(f'  receiver0 -> switch port: {sw_port_r0}')
print(f'  receiver1 -> switch port: {sw_port_r1}')
print()
print('If the dev_port values below are incorrect, SSH into the switch')
print('and run: $SDE/run_bfshell.sh -b then query port.port_hdl_info')
print()

# You may need to set these manually after inspecting the switch.
# Common Tofino 2 dev_port values: 128, 136, 144, 152, ...
DEV_PORT_R0 = 144   # Switch port connected to receiver port 0
DEV_PORT_R1 = 152   # Switch port connected to receiver port 1

# ---- Generate bfrt_python setup script ----
# All 10 senders share the receiver's MAC as dst_mac, so one rule
# per receiver MAC is sufficient.

bfrt_script = f'''# Auto-generated bfrt_python setup script for DiDAQt experiment
# Run with: $SDE/run_bfshell.sh -b /tmp/setup_rules.py

p4 = bfrt.l2_forward.Ingress

# Forward traffic destined for receiver port 0
# vlan_id={sw_vlan_r0} (0 means untagged / L2Bridge)
p4.l2_forward.add_with_forward(
    dst_addr={mac_to_int(r0_mac)},
    port={DEV_PORT_R0},
    vlan_id={sw_vlan_r0}
)
print("Rule added: dst_mac={r0_mac} -> port {DEV_PORT_R0} vlan {sw_vlan_r0}")

# Forward traffic destined for receiver port 1
# vlan_id={sw_vlan_r1} (0 means untagged / L2Bridge)
p4.l2_forward.add_with_forward(
    dst_addr={mac_to_int(r1_mac)},
    port={DEV_PORT_R1},
    vlan_id={sw_vlan_r1}
)
print("Rule added: dst_mac={r1_mac} -> port {DEV_PORT_R1} vlan {sw_vlan_r1}")
'''

print('--- bfrt_python script ---')
print(bfrt_script)

# Upload and execute on the switch
import tempfile, os
with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
    f.write(bfrt_script)
    local_script = f.name

p4_switch.upload_file(local_script, '/tmp/setup_rules.py')
os.unlink(local_script)

print('\nInstalling forwarding rules on switch...')
stdout, stderr = p4_switch.execute(
    f'{SDE}/run_bfshell.sh -b /tmp/setup_rules.py',
    quiet=True, timeout=30
)
print(stdout)
print('Forwarding rules installed.')

Switch port mapping (verify these on the switch):
  sender0   -> switch port: 1
  sender1   -> switch port: 2
  receiver0 -> switch port: 3
  receiver1 -> switch port: 4

If the dev_port values below are incorrect, SSH into the switch
and run: $SDE/run_bfshell.sh -b then query port.port_hdl_info

--- bfrt_python script ---
# Auto-generated bfrt_python setup script for DiDAQt experiment
# Run with: $SDE/run_bfshell.sh -b /tmp/setup_rules.py

p4 = bfrt.l2_forward.Ingress

# Forward traffic destined for receiver port 0
# vlan_id=0 (0 means untagged / L2Bridge)
p4.l2_forward.add_with_forward(
    dst_addr=4670558742048,
    port=144,
    vlan_id=0
)
print("Rule added: dst_mac=04:3F:72:FE:A6:20 -> port 144 vlan 0")

# Forward traffic destined for receiver port 1
# vlan_id=0 (0 means untagged / L2Bridge)
p4.l2_forward.add_with_forward(
    dst_addr=4670558742049,
    port=152,
    vlan_id=0
)
print("Rule added: dst_mac=04:3F:72:FE:A6:21 -> port 152 vlan 0")


Installing forwarding rules on s

## 9. Configure Node Interfaces

Bring up the in-band (L2) interfaces.  When VLAN tagging is active
(L2PTP / cross-site), VLAN offloading is disabled so the sender and
receiver handle tags in software.
The L3 (FABNetv4) interfaces are auto-configured by FABRIC.

In [21]:
vlan_active = s0_vlan > 0  # True when L2PTP (cross-site) is in use

# ---- Sender: bring up interfaces ----
for iface, name in [(s0_os, 'sender port 0'), (s1_os, 'sender port 1')]:
    cmds = f'sudo ip link set {iface} up'
    if vlan_active:
        cmds += f' && sudo ethtool -K {iface} txvlan off rxvlan off'
    sender_node.execute(cmds, quiet=True)
    extra = ', VLAN offload disabled' if vlan_active else ''
    print(f'  {name} ({iface}): up{extra}')

# ---- Receiver: bring up in-band interfaces ----
for iface, name in [(r0_os, 'receiver port 0'), (r1_os, 'receiver port 1')]:
    cmds = f'sudo ip link set {iface} up'
    if vlan_active:
        cmds += f' && sudo ethtool -K {iface} txvlan off rxvlan off'
    receiver_node.execute(cmds, quiet=True)
    extra = ', VLAN offload disabled' if vlan_active else ''
    print(f'  {name} ({iface}): up{extra}')

# ---- L3 interfaces are auto-configured by FABRIC ----
# Verify connectivity
print(f'\nVerifying L3 connectivity: receiver -> controller ({ctrl_ip})...')
stdout, stderr = receiver_node.execute(
    f'ping -c 2 -W 2 {ctrl_ip}',
    quiet=True
)
print(stdout)
print('Interface configuration complete.')

  sender port 0 (enp7s0np0): up
  sender port 1 (enp8s0np0): up
  receiver port 0 (enp8s0np0): up
  receiver port 1 (enp9s0np0): up

Verifying L3 connectivity: receiver -> controller (fe80::bc25:78ff:fe07:3180)...
PING fe80::bc25:78ff:fe07:3180(fe80::bc25:78ff:fe07:3180) 56 data bytes

--- fe80::bc25:78ff:fe07:3180 ping statistics ---
2 packets transmitted, 0 received, 100% packet loss, time 1019ms


Interface configuration complete.


## 10. SSH Access

Use these commands to SSH into each node from your local terminal.

In [16]:
print('='*70)
print('SSH Commands')
print('='*70)
for node in [sender_node, receiver_node, controller_node, p4_switch]:
    name = node.get_name()
    ssh_cmd = node.get_ssh_command()
    print(f'\n--- {name} ---')
    print(f'  {ssh_cmd}')
print()

SSH Commands

--- sender ---
  ssh -i /home/fabric/work/fabric_config/fpga-silver -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:1948:417:7:f816:3eff:fe3a:8ef4

--- receiver ---
  ssh -i /home/fabric/work/fabric_config/fpga-silver -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:1948:417:7:f816:3eff:fe30:84ac

--- controller ---
  ssh -i /home/fabric/work/fabric_config/fpga-silver -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:1948:417:7:f816:3eff:fe38:1a14

--- p4_switch ---
  ssh -i /home/fabric/work/fabric_config/fpga-silver -F /home/fabric/work/fabric_config/ssh_config fabric@2001:1948:417:7:290:fbff:fe76:d00e



## 11. Run the Experiment

The cells below print the exact commands to run on each node.
Open SSH sessions to each node (Section 10) and paste these commands.

**Start order:**
1. Controller (heartbeat monitor)
2. Receiver (both instances)
3. Sender (10 instances)

In [22]:
HB_PORT = 9000

# Build the sender VLAN argument: only pass it when vlan_id > 0.
# With vlan_id=0 (L2Bridge), the sender omits the VLAN tag entirely.
sender_vlan_arg = f' {s0_vlan}' if s0_vlan else ''

print('='*70)
print('STEP 1: Start heartbeat monitor on CONTROLLER')
print('='*70)
print(f'''
cd {REMOTE_DIR}
sudo ./build/heartbeat_monitor {HB_PORT}
''')

print('='*70)
print('STEP 2: Start receiver instances on RECEIVER')
print('='*70)
print(f'''
cd {REMOTE_DIR}

# Receiver instance 0: listens on port 0, heartbeats to controller
sudo ./build/receiver {r0_os} 0 {ctrl_ip} {HB_PORT} &

# Receiver instance 1: listens on port 1, heartbeats to controller
sudo ./build/receiver {r1_os} 1 {ctrl_ip} {HB_PORT} &
''')

print('='*70)
print('STEP 3: Start 10 sender instances on SENDER')
print('='*70)
# All 10 senders use sender port 0, sending to receiver port 0's MAC.
# The switch forwards based on dst_mac to the correct receiver port.
print(f'''
cd {REMOTE_DIR}

# All 10 senders target receiver port 0 via sender port 0.
# Each sender gets a unique sender_id (1-10).
for i in $(seq 1 10); do
    sudo ./build/sender {s0_os} {r0_mac} $i{sender_vlan_arg} &
done

# To stop all senders:
# sudo killall sender
''')

if sender_vlan_arg:
    print(f'(Frames are VLAN-tagged with VID {s0_vlan})')
else:
    print('(Frames are untagged — L2Bridge / same-site)')

print()
print('='*70)
print('EXPECTED OUTPUT')
print('='*70)
print('''
On the controller, you should see heartbeat messages like:

  [14:30:01.234] HB #1 from 10.x.x.x | receiver 0: 10 sender(s) healthy: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
  [14:30:01.334] HB #2 from 10.x.x.x | receiver 0: 10 sender(s) healthy: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

Receiver 1 will send heartbeats with 0 senders (idle, ready for fail-over):

  [14:30:01.234] HB #3 from 10.x.x.x | receiver 1: 0 sender(s) healthy: [(none)]

On the sender, you should see throughput reports:

  sender 1: 1048576 frames, 9.87 Gbps
  sender 2: 1048576 frames, 9.85 Gbps
  ...
''')

STEP 1: Start heartbeat monitor on CONTROLLER

cd /home/ubuntu/didaqt
sudo ./build/heartbeat_monitor 9000

STEP 2: Start receiver instances on RECEIVER

cd /home/ubuntu/didaqt

# Receiver instance 0: listens on port 0, heartbeats to controller
sudo ./build/receiver enp8s0np0 0 fe80::bc25:78ff:fe07:3180 9000 &

# Receiver instance 1: listens on port 1, heartbeats to controller
sudo ./build/receiver enp9s0np0 1 fe80::bc25:78ff:fe07:3180 9000 &

STEP 3: Start 10 sender instances on SENDER

cd /home/ubuntu/didaqt

# All 10 senders target receiver port 0 via sender port 0.
# Each sender gets a unique sender_id (1-10).
for i in $(seq 1 10); do
    sudo ./build/sender enp7s0np0 04:3F:72:FE:A6:20 $i &
done

# To stop all senders:
# sudo killall sender

(Frames are untagged — L2Bridge / same-site)

EXPECTED OUTPUT

On the controller, you should see heartbeat messages like:

  [14:30:01.234] HB #1 from 10.x.x.x | receiver 0: 10 sender(s) healthy: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
  [14:30:01.334] 

## 12. Cleanup

Delete the slice when you are done with the experiment.

In [ ]:
# Uncomment to delete:
# slice.delete()
# print('Slice deleted.')